## Домашнее задание 8. Retrieval‑Augmented Generation (RAG)

В этом домашнем задании мы сделаем ассистента для кулинарных советов с помощью RAG. При этом всю логику мы напишем с нуля без использования специализированных библиотек. В качестве базы данных мы будем использовать рецепты с сайта "Поваренок.Ру".



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

### Загрузка датасета и модели

Таблица с рецептами хранится в файле `povarenok.csv`. Каждая запись содержит название рецепта, список ингредиентов и сам рецепт. Всего датасет содержит 84130 записей, но в рамках этого задания мы будем работать с первыми 10-ю тысячами. Это сделано для ускорения работы, по желанию вы можете взять больше.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files=['povarenok.csv'], split='train')
dataset = dataset.select(range(10000))
dataset

Ингредиенты хранятся в виде списка, записанного в строку. Преобразуем их в обычный список.

In [ ]:
dataset[0]['ingredients']

In [ ]:
def process(sample):
    sample['ingredients'] = eval(sample['ingredients'])
    return sample

dataset = dataset.map(process)

In [ ]:
lens = np.array([len(dataset[i]['text'].split()) for i in range(len(dataset))])

plt.figure(figsize=(10, 2))
plt.subplot(1, 2, 1)
plt.hist(lens, bins=50)
plt.title('Распределение длин текстов')

plt.subplot(1, 2, 2)
plt.hist(lens[lens < 600], bins=50)
plt.title('Распределение длин текстов короче 600 слов')

plt.show()

По распределению длин текстов видим, что часто они довольно длинные. Поэтому смухлевать не получится и перед векторизацией будет нужно разбивать каждый текст на небольшие куски, чтобы не потерять важную информацию.

In [ ]:
dataset[:2]

Для ассистента мы возьмем модель [`Qwen/Qwen2-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct). Это та же модель, которую мы использовали в практической части урока. Она основана на Трансформере и обучена отвечать на запросы пользователя.

In [ ]:
from transformers import pipeline
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

generation_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2-1.5B-Instruct",
    device=device,
    torch_dtype=torch.float16
)

## Генерация без RAG

Убедимся, что по умолчанию Qwen2 не умеет хорошо отвечать на наши вопросы. Иначе смысла в RAG не было бы.

In [ ]:
query = 'Какие ингредиенты входят в салат оливье?'

In [ ]:
messages = [
    {"role": "user", "content": query},
]

output = generation_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.3)

answer = output[0]['generated_text'][1]['content']

print(answer)

Варианты ответа на этот вопрос могут разнится из-за случайности в генерации, но почти наверное у вас в ответе оказалась куча неверных ингредиентов.

##  Retrieval‑Augmented Generation

Теперь улучшим генерацию с помощью RAG. Для этого мы с нуля напишем все компоненты:
1. Векторную базу данных с быстрым поиском
1. Разделение текста на куски
1. Формирование контекста для модели

<img src="https://i.ibb.co/wrHmnpy/rag.png" alt="drawing" width="700"/>

Начнем с первого пункта, как с самого сложного. Наша векторная база данных будет содержать несколько десятков тысяч кусков текста. Мы хотим, чтобы поиск по ней осуществлялся как можно быстрее. Соответственно, обычный поиск ближайших векторов полным перебором нас не устроит. Вместо этого мы реализуем LSH.

### Locality Sensitive Hashing (LSH)
__Для этой части не нужна GPU.__

Идея LSH довольна проста. Мы будем стараться объединять похожие векторы в группы так, чтобы при поиске нужно было перебрать объекты только одной группы. Для этого мы введем несколько хеш-функций. Каждая их них будет строиться по следующему алгоритму:
1. Сгенерируем $k$ случайных векторов (векторы нормали гиперплоскости)
1. Для отдельной точки (вектора) в базе и вектора нормали запишем 1, если точка лежит над соответствующей гиперплоскостью. Для этого угол между вектором нормали и вектором из базы должен быть острым. Другими словами, скалярное произведение между векторами должно быть положительным. Запишем 0 в обратном случае.
1. Повторим процедуру для всех точек и векторов нормали. Так мы для каждой точки получим список из $k$ 0 и 1, который и будет хешем.
1. Запишем в хеш-таблицу найденные точки для каждого хеша.
1. Повторим процедуру 1-4 $L$ раз и получим $L$ разных хеш-таблиц.

Теперь для каждого нового вектора $q$ мы можем очень быстро найти список похожих на него. Для этого посчитаем $L$ хешей для него и отберем все точки, у которых совпал хеш c $q$ хотя бы в одной хеш-таблице. Затем переранжируем найденные точки по расстоянию и выберем нужное число самых близких.

Таким образом, мы очень быстро найдем группу релевантных точек и расстояние будем искать только для них, а не для всех точек в базе.

Сложность формирования хеш-таблиц можно оценить как $O(nLkd)$, где $n$ – число векторов в базе, а $d$ – размерность вектора. В то же время сложность поиска LSH – $O(Lkd + (nLp) \cdot d)$, где $p$ – вероятность того, что хеш двух случайных точек совпадет. Обычно $p$ очень близко к 0, поэтому, мы выигрываем по времени почти в $\frac{n}{Lk}$ раз относительно полного перебора!

<img src="https://i.ibb.co/Qf4Syvx/lsh.png" alt="drawing" width="700"/>

__Задание 1.__ Допишите класс `LSHash`, который реализует векторную базу данных с LSH для быстрого поиска.

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict
from typing import Union


def cosine(x, y):
    return 1 - F.cosine_similarity(x, y, dim=-1)


class LSHash:
    def __init__(self, hidden_size: int, hash_size: int, num_hashtables: int):
        """
        hidden_size: размерность векторов в базе данных
        hash_size: размер хеша, число случайных гиперплоскостей для каждой хеш-таблицы
        num_hashtables: число хеш-таблиц
        """
        self.hidden_size = hidden_size
        self.hash_size = hash_size
        self.num_hashtables = num_hashtables

        self.uniform_planes = [self._generate_random_planes() for _ in range(num_hashtables)]
        self.hash_tables = [defaultdict(list) for i in range(num_hashtables)]

        # По id точки хранит пару (точка, дополнительная информация)
        # Это позволит нам в хеш-таблицах хранить только id для экономии памяти
        self.data_points = dict(
            # id: (point, extra_data)
        )

    def _generate_random_planes(self):
        """
        Генерирует случайные векторы нормали для гиперплоскостей.
        Генерируем из нормального распределения, так как нам важно,
        чтобы углы векторов были распределены равномерно по пространству.
        """
        return torch.randn(self.hash_size, self.hidden_size)

    def _hash(self, planes: torch.Tensor, point: torch.Tensor) -> str:
        """
        Считает хеш для полученной точки (point) по векторам нормали (planes) одной хеш-таблицы.

        planes: Tensor[hash_size, hidden_size]
        point: Tensor[hidden_size]
        
        Возвращает строку из self.hash_size 0 и 1
        """
        
        # ваш код здесь
        pass

    def add_point(self, point_id: Union[int, str], point: torch.Tensor, extra_data=None):
        """
        Добавляет точку и extra_data по id в self.data_points, а так же ее id в каждую из хеш-таблиц.

        point_id: идентификатор точки
        point: точка, которую мы добавим в базу
        extra_data: любые дополнительные данные для точки
        """

        # ваш код здесь
        pass

    def query(self, query_point: torch.Tensor, limit: int = 5) -> dict:
        """
        Находит и возвращает limit штук ближайших к query_point точек из базы данных.
        Сначала в список кандидатов записываются все точки, у которых хеш совпадает с
        query_point хотя бы в одной таблице. Из кандидатов выбираются limit наиболее
        близких к query_point по *косинусу*.

        query_point: Tensor[hidden_size] – входная точка
        limit: int – максимальное число точек на выходе

        Возвращает словарь с полями
            - 'ids': список id найденных точек
            - 'points': Tensor[<= limit, hidden_size] – найденные точки
            - 'extra_data': список дополнительных данных для каждой найденной точки
        """

        # ваш код здесь
        pass

Для проверки, что все работает как надо, построим игрушечный пример на плоскости. Заведем 4 хеш-таблицы с размером хеша 3. Добавим 10 точек в нашу базу данных.

In [ ]:
import uuid

In [ ]:
lsh = LSHash(hash_size=3, hidden_size=2, num_hashtables=4)

for i in range(10):
    point = torch.randn(lsh.hidden_size)
    # нормализация нужна только для визуализации, она не влияет на косинусное растояние
    point /= torch.norm(point)
    # получаем случайный id
    point_id = str(uuid.uuid4())
    lsh.add_point(point_id, point)

Для каждой хеш-таблицы нарисуем векторы нормали, а так же раскрасим одним цветом точки с совпадающими хешами. Можете проверить глазами, что все сработало как надо.

In [ ]:
plt.figure(figsize=(9, 2.5))
# итерируемся по хеш-таблицам
for i in range(lsh.num_hashtables):
    plt.subplot(1, 4, i + 1)

    # рисуем векторы нормали
    planes = lsh.uniform_planes[i]
    for plane in planes:
        plt.plot([0, plane[0]], [0, plane[1]])

    # рисуем точки для каждого хеша
    table = lsh.hash_tables[i]
    for point_ids in table.values():
        group_points = [lsh.data_points[point_id][0] for point_id in point_ids]
        group_points = np.array(group_points)
        plt.scatter(group_points[:, 0], group_points[:, 1])

    plt.title(f'Хеш-таблица {i + 1}')
    plt.xticks([])
    plt.yticks([])
    plt.ylim(-1, 1)
    plt.xlim(-1, 1)

    plt.tight_layout()

Чтобы проверить правильность работы `query`, посмотрим, какие точки оказываются ближайшими при LSH поиске.

In [ ]:
query_point = torch.randn(lsh.hidden_size)
query_point /= torch.norm(query_point)

close_points = lsh.query(query_point, limit=3)['points']

In [ ]:
plt.figure(figsize=(3, 3))

# рисуем все точки в базе данных
all_points = np.array([point[0] for point in lsh.data_points.values()])
plt.scatter(all_points[:, 0], all_points[:, 1], color='black', label='other points')

# рисуем точку запроса
plt.scatter(*query_point, marker='*', s=150, color='green', label='query point')

# рисуем поверх точки, которые оказались ближайшими
plt.scatter(close_points[:, 0], close_points[:, 1], color='red', label='closest points')

plt.legend(loc='center')
plt.xticks([])
plt.yticks([])
plt.xlim(-1, 1)
plt.ylim(-1, 1)

Вы должны увидеть, что ближайшие точки по мнению LSH действительно оказались ближайшими.

### Сравнение с полным перебором

Если верить математике, то поиск для LSH должен работать быстрее полного перебора. Однако пока что не понятно, насколько быстрее. Давайте произведем замеры.

__Задание 2.__ Допишите класс `BruteForceIndex` по аналогии с `LSHash`. При вызове `query` он находит расстояние от `query_point` до каждой точки в базе данных и возвращает `limit` ближайших в нотации `LSHash`.

In [ ]:
class BruteForceIndex:
    def __init__(self):
        self.data_points = dict()

    def add_point(self, point_id: int, point: torch.Tensor, extra_data=None):
        self.data_points[point_id] = (point, extra_data)

    def query(self, query_point: torch.Tensor, limit: int = 5) -> dict:
        """
        С помощью полного перебора находит и возвращает limit штук
        ближайших к query_point точек по косинусному расстоянию.

        query_point: Tensor[hidden_size] – входная точка
        limit: int – максимальное число точек на выходе

        Возвращает словарь с полями
            - 'ids': список id найденных точек
            - 'points': Tensor[<= limit, hidden_size] – найденные точки
            - 'extra_data': список дополнительных данных для каждой найденной точки
        """

        # ваш код здесь
        pass

In [ ]:
hidden_size = 256
n_points = 1000

In [ ]:
bf = BruteForceIndex()
lsh = LSHash(hash_size=5, hidden_size=hidden_size, num_hashtables=10)

for i in range(n_points):
    point = torch.randn(lsh.hidden_size)
    point_id = str(uuid.uuid4())
    lsh.add_point(point_id, point)
    bf.add_point(point_id, point)

In [ ]:
query_point = torch.randn(lsh.hidden_size)

In [ ]:
%%timeit

close_data = bf.query(query_point, limit=5)

In [ ]:
%%timeit

close_data = lsh.query(query_point, limit=5)

Если вы все сделали правильно, то должны быть получить прирост по скорости примерно в 4 раза.

### Разбиение текста на куски

__Для этой части не нужна GPU.__

Так как некоторые тексты довольно длинные, если мы будем кодировать их целиком в вектор, то часть информации потеряется. Вектор все таки имеет ограниченный размер. Чтобы не терять информации, мы поделим текст на куски, а затем каждый кусок будем кодировать отдельно.

Разбивать будем на куски с фиксированным размером слов. Также добавим наложение между кусками, чтобы важная мысль не резалась пополам.

__Задание 3.__ Допишите класс `TextSplitter`. При его вызове он делит текст так, чтобы в каждом куске было `chunk_size` слов. При этом куски текста должны накладываться друг на друга `chunk_overlap` словами. То есть каждый следующий кусок начинается с `chunk_overlap` слов предыдущего. Словом мы называем непрерывную последовательность непробельных символов. Метод `__call__` возвращает список с кусками текстов (строками).

In [ ]:
from typing import List

class TextSplitter:
    def __init__(self, chunk_size: int, chunk_overlap: int):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def __call__(self, text: str) -> List[str]:
        """
        Разбивает текст на куски с фиксированным числом слов и возвращает список полученных кусков
        """
        
        # ваш код здесь
        pass

Размер каждого куска выберем равным 128 слов и добавим наложение в 16 слов.

In [ ]:
text_splitter = TextSplitter(chunk_size=128, chunk_overlap=16)

## RAG

Наконец мы добрались до реализации самого RAG. В этой части нам нужно будет сложить все рецепты из датасета в векторную базу данных, не забыв поделить их на куски, а затем написать код для самой генерации ответов.   
Как и в практической части урока, в качестве эмбеддинговой модели мы возьмем [`intfloat/multilingual-e5-large`](https://huggingface.co/intfloat/multilingual-e5-large).

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large", model_kwargs={'torch_dtype': torch.float16}).to(device)

__Задание 4.__ Напишите функцию `create_vector_db`. Она принимает датасет, модель для кодирования текстов, TextSplitter и гиперпараметры LSH, проходит по всем записям в корпусе, делит тексты на куски, векторизует куски и добавляет полученные точки в созданную LSH базу данных.   
Подумайте о том, какой именно текст вы будете векторизовать, а так же какую информацию вы будете сохранять в базе данных вместе с точкой. Надо ли добавлять в текст название рецепта и ингредиенты или можно ограничиться только самим рецептом?   
Обратите внимание, что `embedding_model.encode` по умолчанию возвращает `np.array`. Чтобы она возвращала тензор, надо установить соответствующий флажок.   
Создание базы данных для 10000 записей с GPU T4 занимает около 10 минут.

In [ ]:
def create_vector_db(dataset, embedding_model, text_splitter, hash_size=10, num_hashtables=30) -> LSHash:
    hidden_size = embedding_model[0].auto_model.config.hidden_size

    # ваш код здесь

In [ ]:
lsh = create_vector_db(dataset, embedding_model, text_splitter, hash_size=10, num_hashtables=30)

Проверим, насколько хорошо ищутся похожие по смыслу тексты.

In [ ]:
query_vector = embedding_model.encode(query, normalize_embeddings=True, device=device, convert_to_tensor=True).cpu().float()

In [ ]:
closest_data = lsh.query(
    query_vector,
    limit=5
)
closest_data

Должен получиться результат, в котором каждый текст относится к Оливье.

__Задание 5.__ Напишите функцию `get_relevant_texts`, которая принимает на вход модель для кодирования текстов, нашу базу данных и строку с запросом и возвращает список релевантных текстов для подачи в контекст модели. Опять же подумайте, какую информацию лучше всего добавлять в эти тексты.

In [ ]:
def get_relevant_texts(embedding_model, lsh: LSHash, query: str, device: torch.device, limit=10) -> List[str]:
    # ваш код здесь
    pass

__Задание 6.__ Напишите функцию `get_answer`, которая принимает на вход модель для генерации `generation_pipeline`, запрос и релевантный контекст. В функции вы должны задать промпт для модели, состоящий из описания задачи, полученных запроса и контекста. После этого верните текст ответа модели для полученного промпта.

In [ ]:
def get_answer(generation_pipeline, query: str, context: str) -> str:
    prompt = ...

    # ваш код здесь
    pass

Теперь, наконец, можно начать тестировать модель, спрашивая у нее всякое! :)

In [ ]:
def predict(query):
    relevant_texts = get_relevant_texts(embedding_model, lsh, query, device)
    context = ' ; '.join(relevant_texts)

    return get_answer(generation_pipeline, query, context)

In [ ]:
print(predict(query))

С RAG результат получается намного лучше. Попробуем еще что-нибудь поспрашивать у модели, не зря же потратили столько времени на реализацию.

In [ ]:
print(predict("Как готовить тирамису?"))

In [ ]:
print(predict("Чем щи отличаются от борща?"))

Как вы должны были заметить, языковая модель с RAG работает куда лучше, чем без него. Несмотря на это, она все равно галлюционирует и регулярно придумывает факты. Качество RAG подхода помимо самих компонент RAG во многом зависит от датасета и модели. Так, датасет с рецептами может не содержать полезных фактов о самих блюдах, а Qwen2 хоть и училась на русском, гораздо лучше справляется с английскими текстами. При построении своего RAG важно учитывать в первую очередь эти факторы, а затем уже экспериментировать с различными наворотами.